# UC11 — Predicción de Severidad en Siniestros de Auto

Predictor de coste de siniestro en FNOL para optimizar reservas técnicas.
**Valor de negocio**: Reducir variabilidad en estimaciones y mejorar dotación de reservas.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS SEVERIDAD_SINIESTROS_DB;
USE SCHEMA SEVERIDAD_SINIESTROS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Siniestros Históricos

In [ ]:
CREATE OR REPLACE TABLE siniestros_historicos AS
SELECT 'SIN' || LPAD(SEQ4(),6,'0') AS siniestro_id,
    CASE MOD(SEQ4(),5) WHEN 0 THEN 'Colision' WHEN 1 THEN 'Alcance' WHEN 2 THEN 'Salida_Via' WHEN 3 THEN 'Robo' ELSE 'Vandalismo' END AS tipo_incidente,
    UNIFORM(10,130,RANDOM())::FLOAT AS velocidad_estimada, UNIFORM(1,4,RANDOM())::FLOAT AS num_vehiculos,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.3 THEN 1 ELSE 0 END AS hay_lesiones,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Turismo' WHEN 1 THEN 'SUV' ELSE 'Furgoneta' END AS tipo_vehiculo,
    UNIFORM(0,20,RANDOM())::FLOAT AS antiguedad_vehiculo, UNIFORM(5000,50000,RANDOM())::FLOAT AS valor_vehiculo,
    CASE MOD(SEQ4(),3) WHEN 0 THEN 'Urbana' WHEN 1 THEN 'Interurbana' ELSE 'Autopista' END AS zona,
    UNIFORM(0,23,RANDOM())::FLOAT AS hora_accidente,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Seco' WHEN 1 THEN 'Lluvia' WHEN 2 THEN 'Niebla' ELSE 'Hielo' END AS condiciones_clima,
    ROUND(EXP(UNIFORM(4.6,10.8,RANDOM())/1.0),2) AS coste_final
FROM TABLE(GENERATOR(ROWCOUNT=>1500));

---

## Paso 3: Datos del Tomador

In [ ]:
CREATE OR REPLACE TABLE tomadores_severidad AS
SELECT siniestro_id, UNIFORM(18,75,RANDOM())::FLOAT AS edad_conductor, UNIFORM(1,40,RANDOM())::FLOAT AS anos_carnet,
    UNIFORM(1,15,RANDOM())::FLOAT AS bonus_malus, UNIFORM(0,4,RANDOM())::FLOAT AS siniestros_previos_3a, UNIFORM(0,1000,RANDOM())::FLOAT AS franquicia
FROM siniestros_historicos;

---

## Paso 4: Feature Engineering y Discretizar Target

In [ ]:
CREATE OR REPLACE TABLE features_severidad AS
SELECT s.siniestro_id,
    CASE WHEN s.tipo_incidente='Colision' THEN 1.0 ELSE 0.0 END AS es_colision,
    CASE WHEN s.tipo_incidente='Alcance' THEN 1.0 ELSE 0.0 END AS es_alcance,
    CASE WHEN s.tipo_incidente='Salida_Via' THEN 1.0 ELSE 0.0 END AS es_salida_via,
    s.velocidad_estimada, s.num_vehiculos, s.hay_lesiones::FLOAT AS hay_lesiones,
    CASE WHEN s.tipo_vehiculo='Turismo' THEN 1.0 ELSE 0.0 END AS es_turismo,
    s.antiguedad_vehiculo, s.valor_vehiculo,
    CASE WHEN s.zona='Urbana' THEN 1.0 ELSE 0.0 END AS zona_urbana,
    s.hora_accidente,
    CASE WHEN s.condiciones_clima IN ('Lluvia','Niebla','Hielo') THEN 1.0 ELSE 0.0 END AS mal_tiempo,
    t.edad_conductor, t.anos_carnet, t.bonus_malus, t.siniestros_previos_3a, t.franquicia,
    CASE WHEN s.coste_final<1000 THEN 'Leve' WHEN s.coste_final<5000 THEN 'Moderado' WHEN s.coste_final<15000 THEN 'Grave' ELSE 'Muy_Grave' END AS severidad
FROM siniestros_historicos s JOIN tomadores_severidad t ON s.siniestro_id=t.siniestro_id;

---

## Paso 5: Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_severidad SAMPLE(80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_severidad WHERE siniestro_id NOT IN (SELECT siniestro_id FROM entrenamiento);

---

## Paso 6: Entrenar Modelo Multi-Clase

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION predictor_severidad(
    INPUT_DATA=>SYSTEM$REFERENCE('TABLE','entrenamiento'),
    TARGET_COLNAME=>'severidad',
    CONFIG_OBJECT=>{'evaluate':TRUE}
);

---

## Paso 7: Evaluar

In [ ]:
CALL predictor_severidad!SHOW_EVALUATION_METRICS();

In [ ]:
CALL predictor_severidad!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar y Recomendar Reservas

In [ ]:
CREATE OR REPLACE TABLE predicciones_severidad AS
SELECT siniestro_id,
    predictor_severidad!PREDICT(OBJECT_CONSTRUCT(
        'es_colision',es_colision,'es_alcance',es_alcance,'es_salida_via',es_salida_via,
        'velocidad_estimada',velocidad_estimada,'num_vehiculos',num_vehiculos,'hay_lesiones',hay_lesiones,
        'es_turismo',es_turismo,'antiguedad_vehiculo',antiguedad_vehiculo,'valor_vehiculo',valor_vehiculo,
        'zona_urbana',zona_urbana,'hora_accidente',hora_accidente,'mal_tiempo',mal_tiempo,
        'edad_conductor',edad_conductor,'anos_carnet',anos_carnet,'bonus_malus',bonus_malus,
        'siniestros_previos_3a',siniestros_previos_3a,'franquicia',franquicia
    )) AS prediccion,
    prediccion['class']::VARCHAR AS severidad_pred,
    CASE prediccion['class']::VARCHAR WHEN 'Leve' THEN 800 WHEN 'Moderado' THEN 3000 WHEN 'Grave' THEN 10000 ELSE 25000 END AS reserva_recomendada,
    severidad AS severidad_real
FROM test;

SELECT severidad_pred, COUNT(*) AS total, AVG(reserva_recomendada)::INT AS reserva_media FROM predicciones_severidad GROUP BY severidad_pred;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Predicción de Severidad en Siniestros')
df = session.table('predicciones_severidad').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Siniestros', len(df))
with c2: st.metric('Reserva Total', f"{df['RESERVA_RECOMENDADA'].sum():,.0f}€")
with c3: st.metric('Exactitud', f"{(df['SEVERIDAD_PRED']==df['SEVERIDAD_REAL']).mean():.1%}")

st.markdown('---')
st.subheader('Distribución por Severidad')
rc = df['SEVERIDAD_PRED'].value_counts()
st.bar_chart(pd.DataFrame({'Siniestros': rc.values}, index=rc.index))

st.markdown('---')
filtro = st.multiselect('Severidad', ['Leve','Moderado','Grave','Muy_Grave'], default=['Grave','Muy_Grave'])
fdf = df[df['SEVERIDAD_PRED'].isin(filtro)]
st.dataframe(fdf[['SINIESTRO_ID','SEVERIDAD_PRED','RESERVA_RECOMENDADA','SEVERIDAD_REAL']], use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS SEVERIDAD_SINIESTROS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;